In [12]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split

In [13]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

mps


In [14]:
BASE_DIR = "output_dataset"

image_dir = os.path.join(BASE_DIR, "images")
crop_dir = os.path.join(BASE_DIR, "crop_mask")
damage_dir = os.path.join(BASE_DIR, "damage_mask")

files = sorted(os.listdir(image_dir))

In [15]:
train_files, temp = train_test_split(files, test_size=0.3, random_state=42)
val_files, test_files = train_test_split(temp, test_size=0.5, random_state=42)

In [16]:
class CropDataset(Dataset):
    def __init__(self, files, transform=None):
        self.files = files
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        f = self.files[idx]

        img = cv2.cvtColor(cv2.imread(os.path.join(image_dir, f)), cv2.COLOR_BGR2RGB)
        img = img.astype(np.float32) / 255.0

        crop = cv2.imread(os.path.join(crop_dir, f), 0)
        damage = cv2.imread(os.path.join(damage_dir, f), 0)

        crop = (crop > 127).astype(np.float32)
        damage = (damage > 127).astype(np.float32)

        damage = damage * crop

        if self.transform:
            aug = self.transform(image=img, masks=[crop, damage])
            img = aug["image"]
            crop, damage = aug["masks"]

        return img, crop.unsqueeze(0), damage.unsqueeze(0)

In [17]:
train_transform = A.Compose([
    A.Resize(320,320),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Normalize(mean=(0,0,0), std=(1,1,1)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(320,320),
    A.Normalize(mean=(0,0,0), std=(1,1,1)),
    ToTensorV2()
])

In [18]:
train_dataset = CropDataset(train_files, train_transform)
val_dataset = CropDataset(val_files, val_transform)
test_dataset = CropDataset(test_files, val_transform)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2)

In [19]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    decoder_attention_type="scse"
).to(device)

In [20]:
bce = nn.BCEWithLogitsLoss()

def dice_loss(pred, target):
    pred = torch.sigmoid(pred)
    inter = (pred * target).sum()
    return 1 - (2*inter + 1e-6)/(pred.sum()+target.sum()+1e-6)

def loss_fn(pred, target):
    return bce(pred, target) + dice_loss(pred, target)

In [21]:
def iou_score(pred, target, threshold=0.4):
    pred = torch.sigmoid(pred)
    pred = (pred > threshold).float()

    inter = (pred * target).sum()
    union = pred.sum() + target.sum() - inter

    return (inter + 1e-6)/(union + 1e-6)

In [22]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [24]:
best_iou = 0

for epoch in range(20):
    model.train()
    loss_total = 0

    for img, _, mask in train_loader:
        img, mask = img.to(device), mask.to(device)

        optimizer.zero_grad()
        out = model(img)

        # 🔥 pseudo-label block
        with torch.no_grad():
            pseudo = (torch.sigmoid(out) > 0.8).float()

        new_mask = torch.maximum(mask, pseudo)

        loss = loss_fn(out, new_mask)
        loss.backward()
        optimizer.step()

        loss_total += loss.item()

    # -------- VALIDATION --------
    model.eval()
    val_iou = 0

    with torch.no_grad():
        for img, _, mask in val_loader:
            img, mask = img.to(device), mask.to(device)
            out = model(img)
            val_iou += iou_score(out, mask, threshold=0.4).item()

    val_iou /= len(val_loader)

    # ✅ print INSIDE loop
    print(f"Epoch {epoch+1} | IoU {val_iou:.4f}")

    if val_iou > best_iou:
        best_iou = val_iou
        torch.save(model.state_dict(), "best_model.pth")
        print("✅ Saved best model")

Epoch 1 | IoU 0.3916
✅ Saved best model
Epoch 2 | IoU 0.4326
✅ Saved best model
Epoch 3 | IoU 0.4535
✅ Saved best model
Epoch 4 | IoU 0.4606
✅ Saved best model
Epoch 5 | IoU 0.4712
✅ Saved best model
Epoch 6 | IoU 0.4811
✅ Saved best model
Epoch 7 | IoU 0.4701
Epoch 8 | IoU 0.4987
✅ Saved best model
Epoch 9 | IoU 0.4948
Epoch 10 | IoU 0.4968
Epoch 11 | IoU 0.4817
Epoch 12 | IoU 0.4949
Epoch 13 | IoU 0.4867
Epoch 14 | IoU 0.4995
✅ Saved best model
Epoch 15 | IoU 0.5139
✅ Saved best model
Epoch 16 | IoU 0.4986
Epoch 17 | IoU 0.5037
Epoch 18 | IoU 0.5239
✅ Saved best model
Epoch 19 | IoU 0.5161
Epoch 20 | IoU 0.5144
